<a href="https://colab.research.google.com/github/VTKhanhLinh/TH-TTNT/blob/main/Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import heapq

# Với 15-puzzle thì n = 4, với 24-puzzle thì n = 5
n = 4

# Định nghĩa các hướng di chuyển: Xuống, Trái, Trên, Phải
rows = [1, 0, -1, 0]
cols = [0, -1, 0, 1]

class Node:
    def __init__(self, parent, mats, empty_pos, h_cost, g_level):
        self.parent = parent
        self.mats = mats
        self.empty_pos = empty_pos
        self.h_cost = h_cost  # Giá trị Heuristic h(n)
        self.g_level = g_level  # Chi phí thực tế g(n)

    def __lt__(self, nxt):
        # Thuật giải AKT dựa trên f(n) = g(n) + h(n)
        return (self.g_level + self.h_cost) < (nxt.g_level + nxt.h_cost)

def calculate_manhattan(mats, goal_pos):
    """Tính tổng khoảng cách Manhattan của tất cả các ô đến vị trí đích"""
    dist = 0
    for r in range(n):
        for c in range(n):
            val = mats[r][c]
            if val != 0:
                target_r, target_c = goal_pos[val]
                dist += abs(r - target_r) + abs(c - target_c)
    return dist

def print_path(node):
    """In ngược từ đích về đầu để xem các bước giải"""
    if node is None:
        return
    print_path(node.parent)
    for row in node.mats:
        print(" ".join(f"{x:2d}" for x in row))
    print(f"--- Bước thứ: {node.g_level} (h={node.h_cost}) ---\n")

def solve_akt(initial, empty_pos, final):
    # Tạo bản đồ vị trí đích để tính Manhattan cho nhanh
    goal_pos = {final[r][c]: (r, c) for r in range(n) for c in range(n)}

    pq = []
    # Khởi tạo nút gốc
    start_node = Node(None, initial, empty_pos, calculate_manhattan(initial, goal_pos), 0)
    heapq.heappush(pq, start_node)

    # Tập visited để tránh lặp lại trạng thái cũ (chặn vòng lặp vô tận)
    visited = set()

    while pq:
        curr = heapq.heappop(pq)

        # Kiểm tra nếu đã đến đích
        if curr.h_cost == 0:
            print("--- TÌM THẤY LỜI GIẢI! CÁC BƯỚC THỰC HIỆN ---")
            print_path(curr)
            return

        # Lưu trạng thái hiện tại vào visited
        state = tuple(map(tuple, curr.mats))
        if state in visited:
            continue
        visited.add(state)

        # Duyệt qua các hướng di chuyển của ô trống (0)
        x, y = curr.empty_pos
        for i in range(4):
            nx, ny = x + rows[i], y + cols[i]
            if 0 <= nx < n and 0 <= ny < n:
                # Tạo ma trận mới bằng cách hoán vị
                new_mats = [list(row) for row in curr.mats]
                new_mats[x][y], new_mats[nx][ny] = new_mats[nx][ny], new_mats[x][y]

                # Tạo node con mới
                child = Node(curr, new_mats, (nx, ny),
                            calculate_manhattan(new_mats, goal_pos), curr.g_level + 1)
                heapq.heappush(pq, child)

# --- THIẾT LẬP BÀI TOÁN ---
# 0 đại diện cho ô trống
initial_state = [
    [1, 2, 3, 4],
    [5, 6, 0, 8],
    [9, 10, 7, 11],
    [13, 14, 15, 12]
]

final_state = [
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 0]
]

# Vị trí dòng 1, cột 2 là số 0 trong initial_state
start_empty_pos = (1, 2)

solve_akt(initial_state, start_empty_pos, final_state)

--- TÌM THẤY LỜI GIẢI! CÁC BƯỚC THỰC HIỆN ---
 1  2  3  4
 5  6  0  8
 9 10  7 11
13 14 15 12
--- Bước thứ: 0 (h=3) ---

 1  2  3  4
 5  6  7  8
 9 10  0 11
13 14 15 12
--- Bước thứ: 1 (h=2) ---

 1  2  3  4
 5  6  7  8
 9 10 11  0
13 14 15 12
--- Bước thứ: 2 (h=1) ---

 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  0
--- Bước thứ: 3 (h=0) ---

